# Week 6 — Data Cleaning, Leakage Fixes & Retraining

Continues from `03_baseline_model.ipynb` and `04_model_comparison.ipynb`. This notebook addresses the open TODOs and Slack feedback before trusting the R² numbers:

- **Leakage columns removed:** `ListPrice`, `OriginalListPrice` (per Aidan's message — listing agents set these using comparables/market info highly correlated with `ClosePrice`, and they don't exist for off-market properties, which is a primary use case). Also removing `DaysOnMarket` and `PurchaseContractDate` (per Ahyo's question — both are only knowable *after* a sale happens, i.e. after `ClosePrice` is effectively determined, so they leak post-outcome information into training).
- **Data quality checks:** duplicates, numeric columns with zero/negative values where that's not physically meaningful (e.g. `LivingArea`, `LotSizeArea`).
- **Outlier filtering:** compute 0.5th/99.5th percentile `ClosePrice` thresholds from the *training set only*, then apply those frozen thresholds to both train and test (no leakage of test distribution into the filter).
- **Re-run training** across Linear Regression, Decision Tree, and Random Forest, and across multiple `month` windows (1, 3, 6, 12) to check how sensitive performance is to the training window size.


Locational Features 
— Distance to CBD, major employment centers, or top-rated schools computed from latitude/longitude. 
5
IDX Exchange  |  AVM Data Science Best Practices  |  Confidential 
— Neighborhood or ZIP-level aggregates (e.g., median price per sqft in the last 12 months), computed on 
training data only and joined forward to avoid leakage. 
— Geohash or spatial clustering as a categorical feature when neighborhood labels are noisy or inconsistent. 
Temporal Features 
— Month/season encoded with sine/cosine transforms to capture cyclical seasonality without an artificial 
ordinal jump from December to January. 
— Property age at time of sale, not just year built, so the feature reflects condition-relevant aging. 
Categorical Encoding 
— For high-cardinality categoricals (neighborhood, subtype), avoid naive one-hot encoding blowing up 
dimensionality; use target encoding computed with cross-validation folds so a category's encoding never 
sees its own row's label — a common, subtle leakage source.

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error


## 1. Load data

In [2]:
df = pd.read_csv("data/df_cleaned.csv")
print(df.shape)
df.head()


(794271, 58)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year,Month,YearMonth,BuyerAgentAOR,ListAgentAOR
0,"Carpet,Wood",True,True,98000.0,556366533,michellefsellsoc@gmail.com,2022-02-25,95000.0,Michelle,Fairchild,...,0.0,ABC Unified,92708,0.0,7284.0,2022,1,2022-01,Unknown,Unknown
1,Unknown,False,False,1200.0,556366530,dineshcalre@gmail.com,2022-02-19,1200.0,DINESH,MAYANI,...,1.0,Apple Valley Unified,92308,0.0,43000.0,2022,1,2022-01,Unknown,Unknown
2,Unknown,True,False,1100000.0,556366044,cindydavishomes@gmail.com,2022-04-15,1100000.0,Cindy,Davis,...,1.0,Solana Beach,92075,370.0,7284.0,2022,1,2022-01,Unknown,Unknown
3,Unknown,True,False,2499999.0,556365765,bryanmeathe@gmail.com,2022-01-04,2499999.0,Bryan,Meathe,...,2.0,Carlsbad Unified,92008,140.0,13376.0,2022,1,2022-01,Unknown,Unknown
4,"Carpet,Tile",Unknown,Unknown,598888.0,556365290,steven@westsideres.com,2022-01-12,640000.0,Steven,Larson,...,1.0,Other,95111,300.0,2738.0,2022,1,2022-01,Unknown,Unknown


## 2. Data quality checks

Before doing anything else: duplicates, and numeric columns with 0 or negative values in fields where that isn't physically meaningful (a house can't have 0 sqft living area or a negative lot size, for example — those are probably data errors, not legitimate values).

In [3]:
# Duplicates
n_dupes = df.duplicated().sum()
print(f"Full-row duplicates: {n_dupes}")

# Drop exact duplicate rows
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f"Shape after dropping duplicates: {df.shape}")


Full-row duplicates: 148
Shape after dropping duplicates: (794123, 58)


In [4]:
# Numeric columns with 0 or negative values
numeric_cols_check = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

zero_or_neg_summary = {}
for c in numeric_cols_check:
    n_zero = (df[c] == 0).sum()
    n_neg = (df[c] < 0).sum()
    if n_zero > 0 or n_neg > 0:
        zero_or_neg_summary[c] = {"zero_count": n_zero, "negative_count": n_neg}

zero_or_neg_df = pd.DataFrame(zero_or_neg_summary).T.sort_values("negative_count", ascending=False)
zero_or_neg_df


,zero_count,negative_count
Longitude,69,793974
ParkingTotal,83861,157
DaysOnMarket,30601,145
Latitude,69,11
OriginalListPrice,11,0
LivingArea,637,0
ClosePrice,39,0
ListPrice,8,0
LotSizeAcres,16953,0
StreetNumberNumeric,10957,0


In [5]:
# Fields where 0/negative are physically implausible and likely data errors.
# Adjust this list after reviewing zero_or_neg_df above -- not every numeric column
# with a zero is necessarily wrong (e.g. a count field can legitimately be 0).
implausible_if_nonpositive = ["LivingArea", "LotSizeArea", "ClosePrice"]

for col in implausible_if_nonpositive:
    if col in df.columns:
        before = len(df)
        df = df[df[col] > 0]
        print(f"{col}: dropped {before - len(df)} rows with value <= 0")


LivingArea: dropped 637 rows with value <= 0
LotSizeArea: dropped 17314 rows with value <= 0
ClosePrice: dropped 38 rows with value <= 0


## 3. Drop leakage columns

Per Aidan's Slack message: `ListPrice` and `OriginalListPrice` are set by listing agents using comparables and market conditions highly correlated with `ClosePrice`, and neither exists for off-market properties (one of the model's primary use cases) — so keeping them both leaks the target and breaks the off-market use case.

Per Ahyo's follow-up: `DaysOnMarket` and `PurchaseContractDate` are only knowable *after* a sale occurs (you don't know how long a property was on the market, or when the contract was signed, until the sale has happened) — so both are also post-outcome information and get dropped alongside `ListPrice`/`OriginalListPrice`.

In [6]:
LEAKAGE_COLS = ["ListPrice", "OriginalListPrice", "DaysOnMarket", "PurchaseContractDate"]

present = [c for c in LEAKAGE_COLS if c in df.columns]
missing = [c for c in LEAKAGE_COLS if c not in df.columns]
print("Dropping:", present)
if missing:
    print("Not found in df (already absent):", missing)

df = df.drop(columns=present)

assert len(df) > 0, "df is empty after dropping leakage columns -- check upstream filters."


Dropping: ['ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'PurchaseContractDate']


## 4. Split (unchanged from `03_baseline_model.ipynb` / `04_model_comparison.ipynb`)

In [7]:
def split(df, month):
    assert len(df) > 0, "split() received an empty dataframe -- check filters applied before this call."
    df["month-year"] = pd.to_datetime(df["YearMonth"])
    df = df.sort_values("month-year")
    assert df["month-year"].notna().any(), "YearMonth failed to parse -- all values are NaT. Check the column's format/content."
    latest_month = df["month-year"].max()
    print("Test month:", latest_month)

    test_df = df[df["month-year"] == latest_month]

    train_start = latest_month - pd.DateOffset(months=month)
    train_df = df[
        (df["month-year"] < latest_month) &
        (df["month-year"] >= train_start)
    ]

    print("Training period:", train_df["month-year"].min(), "to", train_df["month-year"].max())
    print("Testing period:", test_df["month-year"].min(), "to", test_df["month-year"].max())
    assert len(train_df) > 0, "train_df is empty after the split -- try a larger `month` window or check date range."
    assert len(test_df) > 0, "test_df is empty after the split -- check that YearMonth has a valid latest month."
    return train_df, test_df


## 5. Outlier filtering on `ClosePrice`

Compute the 0.5th and 99.5th percentile thresholds using the **training set only**, then apply those *same, frozen* thresholds to the test set. This avoids leaking any information about the test set's price distribution into the filter, while still knocking out extreme outliers on both sides.

In [8]:
def filter_outliers(train_df, test_df, target="ClosePrice", lower_q=0.005, upper_q=0.995):
    assert len(train_df) > 0, "filter_outliers() received an empty train_df."
    lower_thresh = train_df[target].quantile(lower_q)
    upper_thresh = train_df[target].quantile(upper_q)
    print(f"Frozen thresholds from train: [{lower_thresh:,.0f}, {upper_thresh:,.0f}]")

    train_before, test_before = len(train_df), len(test_df)

    train_df = train_df[(train_df[target] >= lower_thresh) & (train_df[target] <= upper_thresh)]
    test_df = test_df[(test_df[target] >= lower_thresh) & (test_df[target] <= upper_thresh)]

    print(f"Train: {train_before} -> {len(train_df)} rows")
    print(f"Test:  {test_before} -> {len(test_df)} rows")
    assert len(train_df) > 0, "All training rows removed by outlier filter -- check ClosePrice values/quantiles."
    assert len(test_df) > 0, "All test rows removed by outlier filter -- thresholds from train may not cover the test month's prices."
    return train_df, test_df


## 6. Prepare data (unchanged)

In [9]:
def prepare_data(train_df, test_df):
    target = "ClosePrice"
    X_train = train_df.drop(columns=[target, "month-year", "YearMonth"])
    y_train = train_df[target]

    X_test = test_df.drop(columns=[target, "month-year", "YearMonth"])
    y_test = test_df[target]

    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    return X_train, y_train, X_test, y_test, numeric_cols, categorical_cols


## 7. Metrics + reusable train/eval (from `04_model_comparison.ipynb`)

In [10]:
def compute_metrics(y_test, y_pred):
    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "mape": mean_absolute_percentage_error(y_test, y_pred),
        "mdape": np.median(np.abs((y_test - y_pred) / y_test))
    }


def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)
    metrics["model"] = model_name

    print(f"{model_name:20s} | R\u00b2: {metrics['r2']:.4f} | MAE: {metrics['mae']:,.0f} | "
          f"MAPE: {metrics['mape']:.2%} | MdAPE: {metrics['mdape']:.2%}")

    return metrics, pipeline


## 8. End-to-end run for a given training window

Wraps split -> outlier filtering (frozen thresholds from train) -> prepare_data -> train all three models, and returns a comparison table.

In [11]:
def run_pipeline(df, month):
    print(f"\n{'='*60}\nmonth = {month}\n{'='*60}")
    train_df, test_df = split(df.copy(), month)
    train_df, test_df = filter_outliers(train_df, test_df)
    X_train, y_train, X_test, y_test, numeric_cols, categorical_cols = prepare_data(train_df, test_df)

    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1),
    }

    results = []
    fitted_pipelines = {}
    for name, model in models.items():
        metrics, pipeline = train_and_evaluate(
            model, name, X_train, y_train, X_test, y_test, numeric_cols, categorical_cols
        )
        metrics["month"] = month
        results.append(metrics)
        fitted_pipelines[name] = pipeline

    comparison_df = pd.DataFrame(results)[["month", "model", "r2", "mae", "mape", "mdape"]]
    comparison_df = comparison_df.sort_values("r2", ascending=False).reset_index(drop=True)
    return comparison_df, fitted_pipelines


In [12]:
comparison_df, fitted_pipelines = run_pipeline(df, month=6)
comparison_df



month = 6
Test month: 2026-05-01 00:00:00
Training period: 2025-11-01 00:00:00 to 2026-04-01 00:00:00
Testing period: 2026-05-01 00:00:00 to 2026-05-01 00:00:00
Frozen thresholds from train: [1,495, 7,200,000]
Train: 117711 -> 116542 rows
Test:  22845 -> 22644 rows
Linear Regression    | R²: 0.6775 | MAE: 322,121 | MAPE: 1987.01% | MdAPE: 34.99%
Decision Tree        | R²: 0.6103 | MAE: 314,570 | MAPE: 161.33% | MdAPE: 35.52%


KeyboardInterrupt: 

## 9. Sensitivity check across training windows

Per the baseline TODO: test different training window sizes (`month=1,3,6,12`) to see whether performance -- and the ranking between models -- holds up, or whether it's an artifact of one particular window.

In [ ]:
all_results = []
for m in [1, 3, 6, 12]:
    cdf, _ = run_pipeline(df, month=m)
    all_results.append(cdf)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.sort_values(["month", "r2"], ascending=[True, False]).reset_index(drop=True)


## 10. Feature importance sanity check (tree-based models, month=6 run)

Confirms the leakage columns no longer show up, and gives a read on what the models are actually keying off of now.

In [ ]:
def get_feature_names(pipeline):
    return pipeline.named_steps["preprocessing"].get_feature_names_out()

for name in ["Decision Tree", "Random Forest"]:
    pipeline = fitted_pipelines[name]
    feature_names = get_feature_names(pipeline)
    importances = pipeline.named_steps["model"].feature_importances_

    top_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(10)
    )
    print(f"\nTop 10 features -- {name}")
    print(top_features.to_string(index=False))
